# 2. Tiền xử lý dữ liệu — bản ghép cuối (feature5)

**Ghép:** phần làm sạch của TV1 (`pre-processing1`) + đặc trưng tốt nhất của TV4 (`pre-processing4`).  
**Input:** `exps/data/data_EDA.csv`  
**Output:** `exps/feature5/`

BEGIN

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys; sys.path.insert(0, "../../extra")
from config import *
from common import *
from utils import *
display.clear_output()

In [3]:
MEMBER   = "TV4"      # tên/mã thành viên -> file log riêng
NOTEBOOK = "pre-processing5"
FEATURE_SET = "feature5"   # thư mục output trong exps/
seed_everything()

In [4]:
df, numeric_columns, category_columns = load_eda()
df.shape

(2919, 82)

## 2.1. Làm sạch
**Baseline = cách của code tham khảo:** bỏ 4 cột thiếu nhiều, cột phân loại điền mode. Cột số để NaN — `make_pipeline()` ở stage 4 sẽ điền median **trong từng fold**.

In [5]:
# ===== CÔNG TẮC: bật lần lượt từng bước, mỗi lần Run All rồi ghi kết quả =====
NAN_SEMANTIC = True    # Bước 1: NaN = "không có"
DROP_OUTLIER = True    # Bước 2: loại GrLivArea > 4000
ORDINAL      = True    # Bước 3: mã hóa thứ bậc cột chất lượng
STEP_NAME = 'feature5' # tên hiển thị trong log, sửa theo công tắc đang bật

# Cột phân loại: NaN nghĩa là "không có" (theo data_description.txt)
NONE_COLS = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
# Cột số: không có hầm/gara/ốp tường -> diện tích, số lượng = 0
ZERO_COLS = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']

QUAL = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ORD_MAP = {c: QUAL for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']}
ORD_MAP.update({
    'BsmtExposure': {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4},
    'BsmtFinType1': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'BsmtFinType2': {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6},
    'GarageFinish': {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3},
})

if NAN_SEMANTIC:
    for c in NONE_COLS:
        df[c] = df[c].fillna('None')
    for c in ZERO_COLS:
        df[c] = df[c].fillna(0)
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
    DROP_COLS = []                                   # giữ lại 4 cột thiếu nhiều
else:
    DROP_COLS = ['Alley', 'PoolQC', 'Fence', 'MiscFeature']
df = df.drop(columns=DROP_COLS)

df['MSSubClass'] = df['MSSubClass'].astype(str)       # mã loại nhà, không phải số
cat_cols = [c for c in category_columns if c not in DROP_COLS] + ['MSSubClass']

if ORDINAL:
    for c, mapping in ORD_MAP.items():
        if c in df.columns:
            df[c] = df[c].fillna('None').map(mapping).fillna(0).astype(int)
    cat_cols = [c for c in cat_cols if c not in ORD_MAP]

for c in cat_cols:                                    # còn NaN thật -> mode của train
    df[c] = df[c].fillna(df.loc[df.is_train == 1, c].mode()[0])
print(len(cat_cols), 'cột one-hot | NaN còn lại trong cột phân loại:', df[cat_cols].isna().sum().sum())

30 cột one-hot | NaN còn lại trong cột phân loại: 0


## 2.2. Ngoại lai
Baseline: **chưa** loại.

In [6]:
if DROP_OUTLIER:
    outlier_mask = (df.is_train == 1) & (df.GrLivArea > 4000)
else:
    outlier_mask = pd.Series(False, index=df.index)
print('Loại', outlier_mask.sum(), 'dòng')
df = df[~outlier_mask].reset_index(drop=True)
print('Số dòng train:', df.is_train.sum())

Loại 4 dòng
Số dòng train: 1456


## 2.3. Feature engineering (từ `pre-processing4` của TV4)

In [7]:
df_clean = df.copy()   # bản đã làm sạch, CHƯA có đặc trưng mới -> gốc cho ablation

def total_sf(d):
    d['TotalSF'] = d['TotalBsmtSF'] + d['1stFlrSF'] + d['2ndFlrSF']
    d['TotalPorchSF'] = d['OpenPorchSF'] + d['EnclosedPorch'] + d['3SsnPorch'] + d['ScreenPorch'] + d['WoodDeckSF']
    return d

def baths(d):
    d['TotalBath'] = (d['FullBath'] + 0.5 * d['HalfBath']
                      + d['BsmtFullBath'].fillna(0) + 0.5 * d['BsmtHalfBath'].fillna(0))
    return d

def age(d):
    d['HouseAge'] = (d['YrSold'] - d['YearBuilt']).clip(lower=0)
    d['RemodAge'] = (d['YrSold'] - d['YearRemodAdd']).clip(lower=0)
    d['IsRemodeled'] = (d['YearRemodAdd'] != d['YearBuilt']).astype(int)
    return d

def flags(d):
    d['HasPool'] = (d['PoolArea'] > 0).astype(int)
    d['Has2ndFloor'] = (d['2ndFlrSF'] > 0).astype(int)
    d['HasGarage'] = (d['GarageArea'].fillna(0) > 0).astype(int)
    d['HasBsmt'] = (d['TotalBsmtSF'].fillna(0) > 0).astype(int)
    d['HasFireplace'] = (d['Fireplaces'] > 0).astype(int)
    return d

def interactions(d):
    d['Qual_x_GrLivArea'] = d['OverallQual'] * d['GrLivArea']
    d['Qual_x_TotalSF'] = d['OverallQual'] * (d['TotalBsmtSF'] + d['1stFlrSF'] + d['2ndFlrSF'])
    return d

FE_GROUPS = {'total_sf': total_sf, 'baths': baths, 'age': age,
             'flags': flags, 'interactions': interactions}

def fix_skew(d, threshold=0.75):
    """log1p các cột số lệch mạnh. Skew tính CHỈ trên dòng train."""
    num = [c for c in d.select_dtypes('number').columns if c not in ('Id', 'SalePrice', 'is_train')]
    tr = d[d.is_train == 1]
    sk = tr[num].skew()
    cols = [c for c in num if abs(sk[c]) > threshold and tr[c].min() >= 0 and d[c].nunique() > 2]
    d[cols] = np.log1p(d[cols])
    return d, cols

def build(groups, skew=False):
    """Từ df_clean: thêm các nhóm đặc trưng, (tùy chọn) giảm skew, one-hot -> x_train, x_test."""
    d = df_clean.copy()
    for g in groups:
        d = FE_GROUPS[g](d)
    if skew:
        d, _ = fix_skew(d)
    X = pd.get_dummies(d.drop(columns=['Id', 'SalePrice']), columns=cat_cols, dtype=int)
    x_tr = X[X.is_train == 1].drop(columns='is_train').reset_index(drop=True)
    x_te = X[X.is_train == 0].drop(columns='is_train').reset_index(drop=True)
    return x_tr, x_te

## 2.4. Dựng feature5 theo cấu hình tốt nhất của ablation & lưu

In [8]:
USE, SKEW = ['total_sf', 'baths', 'interactions'], True   # cấu hình đã chọn ở pre-processing4
x_train, x_test = build(USE, SKEW)
y_train = np.log1p(df_clean.loc[df_clean.is_train == 1, 'SalePrice'].values)
test_id = df_clean.loc[df_clean.is_train == 0, 'Id'].values
assert len(x_train) == len(y_train), (len(x_train), len(y_train))
save_feature_set(FEATURE_SET, x_train, y_train, x_test, test_id,
                 info={'notebook': NOTEBOOK, 'member': MEMBER,
                       'clean': 'TV1: NaN theo ngữ nghĩa + loại ngoại lai + mã hóa thứ bậc',
                       'groups': USE, 'skew': SKEW})

[save] d:/HocSau/House_Prices/house_prices_project/exps/feature5: x_train (1456, 262), x_test (1459, 262)


## 2.5. Kiểm tra nhanh: so sánh feature0 / feature1 / feature4 / feature5

In [9]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

rows = []
for fs in ['feature0', 'feature1', 'feature4', 'feature5']:
    xtr, ytr, _, _ = load_feature_set(fs)
    row = {'feature_set': fs, 'n_rows': len(xtr), 'n_features': xtr.shape[1]}
    for m_name, m in {'Ridge': Ridge(alpha=10),
                      'GBR': GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                                       max_depth=3, random_state=SEED)}.items():
        scores, _ = evaluate(m, xtr, ytr)
        row[m_name] = round(scores.mean(), 5)
        if fs == FEATURE_SET:
            log_experiment(MEMBER, NOTEBOOK, fs, m_name, scores, note='kiểm tra feature5')
    rows.append(row)
pd.DataFrame(rows)

[load] feature0: x_train (1460, 289), x_test (1459, 289)
[load] feature1: x_train (1456, 257), x_test (1459, 257)
[load] feature4: x_train (1456, 293), x_test (1459, 293)
[load] feature5: x_train (1456, 262), x_test (1459, 262)
[log] Ridge        | feature5/all | CV RMSE = 0.12201 ± 0.00612
[log] GBR          | feature5/all | CV RMSE = 0.11963 ± 0.00811


,feature_set,n_rows,n_features,Ridge,GBR
0,feature0,1460,289,0.14716,0.13024
1,feature1,1456,257,0.12636,0.12205
2,feature4,1456,293,0.12080,0.12128
3,feature5,1456,262,0.12201,0.11963


### Kết luận
- Ghép làm sạch của TV1 với đặc trưng của TV4 cho GBR tốt nhất (0.1196) trong 4 feature set, chọn làm feature set cuối.